# 02 - Development / Hold-out Split (on RAW data)

**Goal:** split the raw data **once**, with a fixed seed. The development set is used for all
exploration that touches the target, binning, WoE, variable selection and model fitting. The hold-out
set is used only for final validation.

**Design choices**
- **Split before cleaning.** Cleaning thresholds are then justified on the development set only; the same
  deterministic `clean()` is applied to both sets in notebook 03.
- **70 / 30**, stratified on `target` (with ~10k bad cases the hold-out still has ~3,000 bad).
- **Duplicate-aware:** fully identical rows are kept together on one side, so an exact copy of a development
  row can never sit in the hold-out set and flatter its performance.
- **Fixed seed (42)**; the hold-out set is locked.

**Limitation:** random (out-of-sample) split, not out-of-time. There is no date column, so out-of-time validation is impossible.

In [1]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
for _p in ("../data/processed", "../outputs/figures", "../outputs/metrics", "../outputs/scorecard"):
    Path(_p).mkdir(parents=True, exist_ok=True)   # Git does not keep empty folders

import pandas as pd
from data_processing import load_raw, split_dev_holdout, SEED, HOLDOUT_FRAC

raw = load_raw("../data/cs-training.csv")
print("Raw data:", raw.shape, "| seed =", SEED, "| hold-out fraction =", HOLDOUT_FRAC)

Raw data: (150000, 11) | seed = 42 | hold-out fraction = 0.3


## 1. Split

In [2]:
dev, hold = split_dev_holdout(raw)
summary = pd.DataFrame({
    "rows": [len(raw), len(dev), len(hold)],
    "bad": [raw.target.sum(), dev.target.sum(), hold.target.sum()],
}, index=["full", "development", "hold-out"])
summary["share_of_rows"] = (summary["rows"] / len(raw)).round(4)
summary["bad_rate"] = (summary["bad"] / summary["rows"]).round(4)
summary

,rows,bad,share_of_rows,bad_rate
full,150000,10026,1.0000,0.0668
development,104987,7020,0.6999,0.0669
hold-out,45013,3006,0.3001,0.0668


## 2. Checks

In [3]:
assert dev.index.intersection(hold.index).empty, "ids overlap"
assert len(dev) + len(hold) == len(raw)
cols = list(raw.columns)
overlap = dev.drop_duplicates().merge(hold.drop_duplicates(), on=cols)
assert overlap.empty, "identical rows found on both sides"
print("OK: no id overlap, row counts add up, no identical row appears in both sets")
print("Duplicated rows inside development:", dev.duplicated().sum(), "| inside hold-out:", hold.duplicated().sum())

OK: no id overlap, row counts add up, no identical row appears in both sets
Duplicated rows inside development: 414 | inside hold-out: 195


In [4]:
# Distributions should look alike: compare medians and missing rates
c = ["age", "revolving_util", "debt_ratio", "monthly_income", "n_open_lines"]
pd.concat({
    "dev_median": dev[c].median(),
    "hold_median": hold[c].median(),
    "dev_missing_%": (dev[c].isna().mean() * 100).round(2),
    "hold_missing_%": (hold[c].isna().mean() * 100).round(2),
}, axis=1).round(3)

,dev_median,hold_median,dev_missing_%,hold_missing_%
age,52.000,52.000,0.00,0.00
revolving_util,0.154,0.154,0.00,0.00
debt_ratio,0.366,0.367,0.00,0.00
monthly_income,5400.000,5379.000,19.86,19.73
n_open_lines,8.000,8.000,0.00,0.00


**Reading:** bad rate, medians and missing rates are almost identical across the two sets (stratified random split).
Row shares are 70/30 up to a tiny deviation because duplicate groups are moved as a block.

## 3. Save

In [5]:
dev.to_csv("../data/processed/raw_dev.csv")
hold.to_csv("../data/processed/raw_holdout.csv")
summary.to_csv("../outputs/metrics/split_summary.csv")
print("Saved raw_dev.csv, raw_holdout.csv, split_summary.csv")

Saved raw_dev.csv, raw_holdout.csv, split_summary.csv


## Summary

- 70/30 stratified, duplicate-aware split of the raw data, seed 42, done once.
- From now on, **everything that looks at the target uses `raw_dev.csv` only**.

Next: notebook 03 - data quality evidence and cleaning rules (development set), then apply to both sets.